# Experiments

Each experiment changes one setting of the current system and is measured against the current baseline on the **tune split**. Every experiment follows the same pattern:

- **Question** — what the experiment should answer, and why it was worth running
- **Runs** — the recorded evaluation runs compared
- **Evidence** — summary and question-by-question comparison
- **Decision** — adopt, reject, or investigate further

Evaluation runs are produced by `scripts/evaluate_litqa2.py`; this notebook only reads their recorded results. A setting is confirmed on the test split only once it has been adopted, and test results are only examined as summaries.

**Outputs and licensing.** Outputs are stripped on commit by `nbstripout`, except for cells marked `keep_output`, which print only counts, metrics and question IDs.

**Current baseline:** hybrid retrieval (BM25 + embeddings, Reciprocal Rank Fusion), `k = 5`, 300-word chunks, Qwen3 8B at temperature 0.

In [ ]:
import json
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
RESULTS = ROOT / "experiments" / "results"


def read_jsonl(path: Path) -> list[dict]:
    with path.open(encoding="utf-8") as handle:
        return [json.loads(line) for line in handle if line.strip()]


def find_run(pattern: str) -> Path:
    """Resolve a run ID or pattern to exactly one results folder."""
    matches = sorted(RESULTS.glob(pattern))
    if len(matches) != 1:
        names = [m.name for m in matches]
        raise ValueError(f"{pattern!r} matched {len(matches)} runs: {names}. Pin the exact run ID.")
    return matches[0]


def load(pattern: str) -> dict:
    run = find_run(pattern)
    return {
        "name": run.name,
        "config": json.loads((run / "config.json").read_text()),
        "summary": json.loads((run / "summary.json").read_text()),
        "rag": {r["id"]: r for r in read_jsonl(run / "results.jsonl") if r["mode"] == "rag"},
    }


METRICS = {
    "rag": ["n", "correct", "wrong", "abstained", "accuracy", "precision",
            "key_passage_hit_rate", "key_passage_checked", "mean_distinct_papers"],
    "source_removed": ["n", "abstained", "wrong", "expected_behaviour_rate"],
}


def compare(runs: dict[str, dict]) -> None:
    """Print the main metrics of several runs side by side."""
    for name, run in runs.items():
        config = run["config"]
        print(
            f"{name}: {run['name']} | retrieval {config.get('retrieval', '(not recorded)')}, "
            f"k {config['k']}, "
            f"commit {config['git']['commit'][:7]}, "
            f"uncommitted changes: {config['git']['uncommitted_changes']}"
        )
    for mode, keys in METRICS.items():
        print(f"\n{mode}")
        print(f"  {'':<24}" + "".join(f"{name:>14}" for name in runs))
        for key in keys:
            values = [run["summary"].get(mode, {}).get(key) for run in runs.values()]
            cells = "".join(
                f"{v:>14.3f}" if isinstance(v, float) else f"{str(v):>14}" for v in values
            )
            print(f"  {key:<24}{cells}")


def changed_questions(before: dict, after: dict) -> None:
    """List questions whose outcome or key-passage hit differs between two runs."""
    print(f"{'id':<10} {'before':<24} {'after':<24}")
    changes = 0
    for qid in sorted(before["rag"]):
        b, a = before["rag"][qid], after["rag"][qid]
        if (b["outcome"], b["key_passage_hit"]) != (a["outcome"], a["key_passage_hit"]):
            changes += 1
            print(
                f"{qid[:8]:<10} {b['outcome']:<10} kp={str(b['key_passage_hit']):<10}"
                f" {a['outcome']:<10} kp={a['key_passage_hit']}"
            )
    print(f"\n{changes} of {len(before['rag'])} questions changed")

---

## Experiment 1 — Number of retrieved chunks (`k`)

**Question.** Does giving the model 8 chunks instead of 5 help, or does it distract?

**Why it was worth running.** Switching to hybrid retrieval cost one tune question its answer passage: keyword-promoted chunks pushed the passage out of the top five. More chunks could recover such cases. But more chunks also put more loosely related text in the prompt, which could lead the model to answer where it should decline — so source-removed abstention matters as much as accuracy.

**Runs (tune split).**

| Label | Retrieval | k | Note |
|---|---|---|---|
| `dense_k5` | dense | 5 | Original baseline |
| `dense_k8` | dense | 8 | Run by accident: the evaluation script still defaulted to dense. Its folder name records the mode; its config does not. Kept, because it is a valid run and separates the effect of *more* passages from *better* passages |
| `hybrid_k5` | hybrid | 5 | Current baseline |
| `hybrid_k8` | hybrid | 8 | The intended experiment |

`closed_book` was not re-run for `k = 8`: it uses no retrieval and has proven deterministic.

In [ ]:
runs = {
    "dense_k5": load("20261021T114958Z_tune"),
    "dense_k8": load("20261023T104645Z_tune_dense"),
    "hybrid_k5": load("*_tune_hybrid"),
    "hybrid_k8": load("20261023T112224Z_tune_hybrid_k8"),
}
compare(runs)

**Question by question — the intended comparison** (hybrid, `k = 5` → `k = 8`):

In [ ]:
changed_questions(runs["hybrid_k5"], runs["hybrid_k8"])

**Question by question — the accidental comparison** (dense, `k = 5` → `k = 8`):

In [ ]:
changed_questions(runs["dense_k5"], runs["dense_k8"])

**Findings.**

| Tune split (38) | dense k=5 | dense k=8 | hybrid k=5 | hybrid k=8 |
|---|---|---|---|---|
| Correct | 32 | 32 | 33 | **34** |
| Wrong | 1 | 1 | 1 | **0** |
| Key-passage hit | 79% | 89% | 89% | **95%** |
| Source-removed abstention | 38/38 | 38/38 | 38/38 | 38/38 |

- **More chunks find more answer passages** with either retrieval method.
- **Better passages matter as much as more passages.** Dense with 8 chunks finds as many answer passages as hybrid with 5, but answers no more questions correctly. Hybrid with 8 chunks gains two more passages and two more correct answers.
- **No loss of caution.** With the source paper removed, the model received 8 loosely related chunks from about four papers per question, and still declined every question.
- **Cost:** about 60% more prompt text per question, so somewhat slower answers.
- **Caveat:** neighbouring configurations differ by one or two questions, at the limit of what 38 questions can show. The result is convincing because every step points the same way and no metric gets worse.

**Decision.** Adopt hybrid retrieval with `k = 8`, provisionally, and confirm it on the test split. Because every test run is another opportunity for chance to influence a decision, later adopted changes will be bundled into a single test confirmation rather than tested one by one.